# Lexicon Sentiment and the Loughran–McDonald Dictionary

A sentiment lexicon is a linear classifier whose weights were fixed before anyone looked at labeled data: $+1$ on
the positive list, $-1$ on the negative list, $0$ elsewhere. This notebook asks what that fixing costs, on Topic 17's
synthetic finance corpus and its tone task. It makes six points:
- a lexicon turns a document into a point $(\hat p_+, \hat p_-)$ of the category simplex, and net tone
  $\hat p_+ - \hat p_-$ projects that point onto one direction; the Bayes rule of a category world cuts the plane by
  lines, and it reduces to net tone only when the tones' log-rate ratios are antisymmetric (Prop 1);
- the sign of the raw score is wrong with probability at most $(1 - (\sqrt a - \sqrt b)^2)^L$, an exponent that is
  exact (Thm 1), and the hit rates of a finance lexicon make documents of 200 tokens barely enough;
- a general-purpose negative list that counts *tax*, *cost*, and *liability* adds tone-independent noise: the mean gap
  survives and the variance grows (Prop 2), Loughran and McDonald's point;
- no lexicon reads "costs rose", "costs exceeded expectations", or (without a scope rule) "not weak" (Def 2);
- a lexicon makes a good prior for logistic regression: shrinking toward it rather than toward zero never falls more
  than 0.01 below the better of the dictionary and plain LR, and beats both from 30 documents on (Thm 2);
- harmonic propagation from four seed words recovers most polarities' signs (Thm 3), but it cannot tell a polar word
  from a topical one.

Every number below is computed by `lexicon_sentiment_loughran_mcdonald.py`, whose harness asserts each claim. The
real Loughran–McDonald dictionary is not committed and is not used here; the asserts run on the corpus's hand-written
stand-in lexicon.

In [ ]:
import sys
from pathlib import Path

import numpy as np

here = Path.cwd()
sys.path.insert(0, str(here if (here / "lexicon_sentiment_loughran_mcdonald.py").exists()
                       else here / "notebooks" / "lexicon-sentiment-loughran-mcdonald"))
import lexicon_sentiment_loughran_mcdonald as ls

ltc, fc = ls.ltc, ls.fc
print({c: len(ls.LEX[c]) for c in ls.CATEGORIES}, "polysemes:", sorted(fc.FINANCE_POLYSEMES))
_, ytr, _, _, _, yte = ltc.fin_split()
print("train / test documents:", len(ytr), len(yte))

## 1. A lexicon is a linear functional

Def 1: a lexicon is a family of disjoint word sets $P, N, \dots$; its weight vector is $w_L = \mathbf 1_P - \mathbf 1_N$,
so the raw score of a count vector $x$ is $w_L \cdot x = \text{pos} - \text{neg}$ and the net tone is
$(\text{pos} - \text{neg})/L$. The document's **category rates** $(\hat p_+, \hat p_-)$ place it in a triangle (the
simplex, with "everything else" as the third corner), and net tone measures it along a single direction of that
triangle.

In [ ]:
ls.test_lexicon_is_a_linear_functional()
for t in ls.TONES:
    print(f"tone {t:+d}: hits per token (pos, neg, unc, lit) =", np.round(ls.tone_rates()[t], 5))

**Prop 1.** Suppose each tone $t$ draws every token's category i.i.d. with rates $q_t = (q_{t+}, q_{t-}, q_{t0})$
(and words within a category from a tone-free distribution). Then (a) the posterior depends on the document only
through its category counts, and $\log P(t \mid x) = \text{pos}\log q_{t+} + \text{neg}\log q_{t-} +
\text{other}\log q_{t0} + c(x)$, which divided by $L$ is affine in $(\hat p_+, \hat p_-)$: the Bayes regions are cut by
lines; (b) a pair of tones is decided by net tone iff $u_+ = -u_-$, where
$u_c = \log(q_{tc}/q_{t'c}) - \log(q_{t0}/q_{t'0})$; (c) in the symmetric-tilt world (positive documents scale positive
words by $e^\beta$ and negative ones by $e^{-\beta}$, equal base mass $m$) the rule is net tone against $\pm\tau$,
$\tau = \log(1 + 2m(\cosh\beta - 1))/\beta$.

In [ ]:
ls.test_prop1_category_world_is_linear()
ls.test_prop1_tilt_world_bayes_rule()
cc = ls.category_classifier()
for pair, (up, un) in cc["slopes"].items():
    print(f"tones {pair}: u+ = {up:+.3f}, u- = {un:+.3f}")
print("net tone with tuned cutpoints:", round(ls.lexicon_classifier()["acc"], 3),
      " category-world rule:", round(cc["acc"], 3),
      " Topic 17's unigram LR:", round(ltc.fin_fit(ltc.FinConfig("lr", False))["acc"], 3))
ls.test_net_tone_discards_a_coordinate()

The corpus is far from antisymmetric: positive documents have more positive words than neutral ones ($u_+ \approx
0.97$) but hardly fewer negative ones ($u_- \approx 0.19$), because a positive filing still reports some bad news.
Neutral documents are the ones with *few polar words of either kind*. Net tone puts them alongside documents with
many polar words in balance; the two coordinates keep them apart.

## 2. How long a document must be

**Thm 1.** Let each token be a positive hit with probability $a$, a negative hit with probability $b < a$, and
neither otherwise, independently, and let $S_L$ be the raw score of $L$ tokens. Then
$P(S_L \le 0) \le \rho^L$ with $\rho = 1 - (\sqrt a - \sqrt b)^2$, and $\frac1L \log P(S_L \le 0) \to \log \rho$.

The bound is Chernoff's: $E e^{-tX} = a e^{-t} + b e^{t} + 1 - a - b$ is minimized at $e^t = \sqrt{a/b}$. Tightness is
Cramér's theorem. We check the bound against the exact trinomial law, and the exponent's convergence.

In [ ]:
ls.test_thm1_bound_holds()
ls.test_thm1_exponent_is_tight()
r = ls.tone_rates()[1]
rho = ls.chernoff_rho(r[0], r[1])
C_te = ls.split_counts()[2]
Lbar = C_te[yte == 2, 4].mean()
print(f"positive documents: a = {r[0]:.4f}, b = {r[1]:.4f}, rho = {rho:.6f}, mean length {Lbar:.1f}")
print(f"bound rho^L = {rho ** Lbar:.3f}; exact i.i.d. error = {ls.sign_error(r[0], r[1], int(round(Lbar))):.3f}; "
      f"measured = {float((ls.net_tone(C_te[yte == 2]) <= 0).mean()):.3f}")
ls.test_length_on_the_corpus()

The bound is loose at this length (about 5.5 expected hits per document), but the i.i.d. model's *exact* error
matches the measured one almost to the third decimal: one in four positive filings scores zero or below.

## 3. Contamination: Loughran and McDonald's point

**Prop 2.** Add a set of words with rate $c$, independent of tone, to the negative list. Per token, the gap between
the two polar tones' means stays $2(a - b)$, while the variances rise by exactly $c(1 - c + 2(a-b))$ (positive
documents) and $c(1 - c - 2(a - b))$ (negative ones), so Fisher's ratio falls. Re-tuning the cutpoints absorbs the
shift of the means, but not the extra variance.

In [ ]:
ls.test_prop2_contamination_moments()
ls.test_general_lexicon_misreads_finance()
for name in ("finance", "general"):
    lc, mo = ls.lexicon_classifier(name), ls.score_moments(name)
    print(f"{name:8} 3-class acc {lc['acc']:.3f}  polar sign acc {lc['sign_acc']:.3f}  Fisher ratio {mo['snr']:.3f}  "
          "means", [round(mo['mean'][t], 4) for t in ls.TONES])
for e in ls.edgar_polysemes():
    print(f"EDGAR {e['company']:26} tokens {e['tokens']}  stand-in negative hits {e['negative']}  polysemes {e['words']}")
ls.test_edgar_has_polysemes()

## 4. What no lexicon sees

Using the corpus's gold aspects, we sort single-polarity sentences by the template that produced them and score the
lexicon's sign on each. **Def 2** (negation scope $k$): a polar hit preceded by a negator within $k$ tokens counts for
the other side.

In [ ]:
ls.test_constructions_a_lexicon_cannot_read()
ls.test_negation_scope()
t0, t1 = ls.construction_table(0), ls.construction_table(1)
for c in ls.CONSTRUCTIONS:
    print(f"{c:22} n={t0[c]['n']:5}  acc {t0[c]['acc']:.3f}  abstain {t0[c]['abstain_share']:.3f}  with k=1: {t1[c]['acc']:.3f}")
ng = ls.negation_sweep()
print("document accuracy by scope k", dict(zip(ng["k"], np.round(ng["acc"], 3))), " false flips", dict(zip(ng["k"], ng["far"])))

Bare directions ("costs rose 5 percent") carry no lexicon word, so the lexicon abstains; for cost-like aspects
"exceeded expectations" is bad news written in a positive word. These are Topic 17's exclusive-or of valence and
direction: no fixed unigram weighting reads them.

Uncertainty and litigious words are not a third polarity. Uncertainty words come at the same rate in every tone, and
litigious words appear only in 8-K filings; the rule "any uncertainty word means neutral" falls well below net tone.

In [ ]:
ls.test_uncertainty_is_not_neutrality()
cp = ls.category_profile()
for t in ls.TONES:
    print(f"tone {t:+d}: per 100 tokens (pos, neg, unc, lit) =", np.round(cp["by_tone"][t], 3))
for dt, v in cp["by_type"].items():
    print(f"{dt:13}: per 100 tokens (pos, neg, unc, lit) =", np.round(v, 3))
print(ls.category_features_accuracy())

## 5. The lexicon as a prior

**Thm 2.** Let $A$ be a $V \times r$ matrix of lexicon directions ($A = w_L$, or $A = [\mathbf 1_P, \mathbf 1_N]$), and
fit softmax LR with logits $x \cdot \Delta_k + (A^\top x)\cdot\beta_k + b_k$ and penalty $\frac\lambda2\|\Delta\|^2$.
(a) Equivalently, the penalty is $\frac\lambda2\sum_k \|P_\perp W_k\|^2$, ridge on the part of each class's weights
orthogonal to $\operatorname{span}(A)$, and $A^\top\Delta_k = 0$ at the optimum. (b) $\|\Delta_k\| \le G/\lambda$ with
$G = \frac1n\sum_i \|x_i\|$. (c) As $\lambda \to \infty$ the fit converges to the softmax on the lexicon counts
$A^\top x$ alone: the dictionary with fitted weights and cutpoints.

In [ ]:
ls.test_thm2_prior_limit()
for kind in ls.KINDS:
    th = ls.thm2_limit(kind)
    print(kind, " G =", round(th["G"], 3))
    for row in th["rows"]:
        print(f"   lambda {row['lam']:>6g}: lambda*max|Delta_k| = {row['lam_norm']:.3f}, "
              f"A'Delta/|.| = {row['orth']:.1e}, max prob gap to the limit = {row['gap']:.4f}")

In [ ]:
ls.test_dictionary_versus_labels()
c = ls.learning_curves()
print("n         ", c["n"])
for m in ls.METHODS:
    print(f"{m:16}", np.round(c[m], 3))
print("first n at which unigram LR matches the dictionary:", c["crossover"])

Read the rows as a price list. The dictionary's two counts are worth about a thousand labeled filings to plain
logistic regression. Shrinking toward the dictionary rather than toward zero never falls more than 0.01 below the better
end, and it beats plain LR at 1,500 documents too (0.766 against 0.739 for the net-tone prior).

## 6. Inducing a lexicon

**Thm 3.** On a weighted graph $W$ with seed set $S$ (values $\pm 1$), if every vertex in $U$ is connected to a seed,
the system $(D - W)_{UU} f_U = W_{US} f_S$ has exactly one solution; it satisfies $\min f_S \le f \le \max f_S$, and
$f_U = 2h - 1$ with $h_v$ the probability that a random walk from $v$ reaches a $+1$ seed before a $-1$ seed (Zhu,
Ghahramani, and Lafferty 2003). We check uniqueness, the maximum principle, convergence of label propagation, and
the random-walk identity on random graphs, then propagate from *strong*, *robust* / *weak*, *adverse* over the
training documents' co-occurrence graph (positive PMI over word types in at least 30 documents).

In [ ]:
ls.test_thm3_harmonic_properties()
ls.test_induction_on_the_corpus()
sc = ls.induction_scores(ls.induced())
print(f"sign accuracy on {sc['n_polar']} polar words: {sc['sign_acc']:.3f}")
print("mean |z| by category:", {k: round(v, 3) for k, v in sc["mean_abs"].items()})
res = ls.induced()
print("polysemes:", {w: round(float(res["z"][res["voc"][w]]), 2) for w in sorted(fc.FINANCE_POLYSEMES) if w in res["voc"]})
print("most positive:", sc["top"])
print("most negative:", sc["bottom"])
rob = np.array(ls.seed_robustness())
print(f"random seed sets: median sign accuracy {np.median(rob):.3f}, 10th percentile {np.quantile(rob, 0.1):.3f}")

Propagation gets the *sign* of a known polar word right nine times in ten. It does not neutralize the polysemes
(half of them, *tax*, *taxes*, *liability*, and *crude*, land on the negative side), and the most extreme scores belong
to sector words that co-occur with the seeds' documents. Induction scores a candidate list well; it cannot choose
the list, which is the curation Loughran and McDonald did by hand.

## 7. The real dictionary (optional)

`load_lm_dictionary()` returns the Loughran–McDonald Master Dictionary in the stand-in's shape if you have a copy
(set `LM_MASTER_DICTIONARY`, or pass `fetch=True` to download it into `~/.cache/formalnlp/lm/`). It is free for
academic research; commercial use needs a license from the authors. Nothing in this notebook's asserts depends on it.

In [ ]:
ls.test_lm_csv_parser()
ls.test_lm_dictionary_not_committed()
lm = ls.load_lm_dictionary()
if lm is None:
    print("No local copy of the Loughran–McDonald dictionary: skipping (see", ls.LM_PAGE, ")")
else:
    print({c: len(v) for c, v in lm.items()})
    for d in fc.load_edgar_excerpts():
        toks = fc._tokens(d.text)
        print(f"{d.company:26}", {c: sum(t in lm[c] for t in toks) for c in ls.CATEGORIES})

## The whole harness

The `.py` runs every check above (and the lab's data mirror) in one go.

In [ ]:
failed = ls._run_harness()
assert failed == 0

## Where this sits

Topic 19 (aspect-based sentiment) attaches polarity to the aspect a sentence evaluates, which is what the
"direction only" and "cost-like verb" rows above need; the capstone scores earnings calls with the dictionary and a
learned model side by side. formalRAG's `vector-space-model-tfidf` covers the tf–idf weighting Loughran and McDonald
also propose; formalStatistics' `large-deviations` proves Cramér's theorem behind Thm 1's exponent.